In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

POSTGRES_JAR = "/home/jovyan/work/spark/jars/postgresql-42.7.4.jar"
CLICKHOUSE_JAR = "/home/jovyan/work/spark/jars/clickhouse-jdbc-0.6.3.jar"

JARS = f"{POSTGRES_JAR},{CLICKHOUSE_JAR}"

POSTGRES_URL = "jdbc:postgresql://postgres:5432/spark"
CLICKHOUSE_URL = "jdbc:clickhouse://clickhouse:8123/reports"

POSTGRES_PROPS = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver",
}

CLICKHOUSE_PROPS = {
    "user": "default",
    "password": "",
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}

spark = (
    SparkSession.builder
    .appName("clickhouse-reports")
    .config("spark.jars", JARS)
    .config("spark.driver.extraClassPath", JARS)
    .config("spark.executor.extraClassPath", JARS)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [2]:
fact_sale = spark.read.jdbc(POSTGRES_URL, "fact_sale", properties=POSTGRES_PROPS)
dim_customer = spark.read.jdbc(POSTGRES_URL, "dim_customer", properties=POSTGRES_PROPS)
dim_pet = spark.read.jdbc(POSTGRES_URL, "dim_pet", properties=POSTGRES_PROPS)
dim_seller = spark.read.jdbc(POSTGRES_URL, "dim_seller", properties=POSTGRES_PROPS)
dim_supplier = spark.read.jdbc(POSTGRES_URL, "dim_supplier", properties=POSTGRES_PROPS)
dim_product = spark.read.jdbc(POSTGRES_URL, "dim_product", properties=POSTGRES_PROPS)
dim_store = spark.read.jdbc(POSTGRES_URL, "dim_store", properties=POSTGRES_PROPS)

print("fact_sale:", fact_sale.count())
print("dim_customer:", dim_customer.count())
print("dim_pet:", dim_pet.count())
print("dim_seller:", dim_seller.count())
print("dim_supplier:", dim_supplier.count())
print("dim_product:", dim_product.count())
print("dim_store:", dim_store.count())

fact_sale: 10000
dim_customer: 10000
dim_pet: 9850
dim_seller: 10000
dim_supplier: 10000
dim_product: 10000
dim_store: 10000


In [3]:
sales = (
    fact_sale.alias("fs")
    .join(dim_customer.alias("c"), F.col("fs.customer_id") == F.col("c.id"), "left")
    .join(dim_product.alias("p"), F.col("fs.product_id") == F.col("p.id"), "left")
    .join(dim_store.alias("st"), F.col("fs.store_id") == F.col("st.id"), "left")
    .join(dim_supplier.alias("su"), F.col("fs.supplier_id") == F.col("su.id"), "left")
    .select(
        F.col("fs.id").alias("sale_id"),
        F.col("fs.date").alias("sale_date"),
        F.year("fs.date").alias("sale_year"),
        F.month("fs.date").alias("sale_month"),
        F.col("fs.quantity").cast("int").alias("quantity"),
        F.col("fs.total_price").cast("double").alias("total_price"),

        F.col("c.id").alias("customer_id"),
        F.col("c.first_name").alias("customer_first_name"),
        F.col("c.last_name").alias("customer_last_name"),
        F.col("c.email").alias("customer_email"),
        F.col("c.country").alias("customer_country"),

        F.col("p.id").alias("product_id"),
        F.col("p.name").alias("product_name"),
        F.col("p.category").alias("product_category"),
        F.col("p.price").cast("double").alias("product_price"),
        F.col("p.rating").cast("double").alias("product_rating"),
        F.col("p.reviews").cast("int").alias("product_reviews"),

        F.col("st.id").alias("store_id"),
        F.col("st.name").alias("store_name"),
        F.col("st.city").alias("store_city"),
        F.col("st.country").alias("store_country"),

        F.col("su.id").alias("supplier_id"),
        F.col("su.name").alias("supplier_name"),
        F.col("su.country").alias("supplier_country"),
    )
)

print("sales:", sales.count())
sales.show(5, truncate=False)

sales: 10000
+-------+----------+---------+----------+--------+-----------+-----------+-------------------+------------------+---------------------------+----------------+----------+------------+----------------+-------------+--------------+---------------+--------+-------------+----------+-------------+-----------+-------------+----------------+
|sale_id|sale_date |sale_year|sale_month|quantity|total_price|customer_id|customer_first_name|customer_last_name|customer_email             |customer_country|product_id|product_name|product_category|product_price|product_rating|product_reviews|store_id|store_name   |store_city|store_country|supplier_id|supplier_name|supplier_country|
+-------+----------+---------+----------+--------+-----------+-----------+-------------------+------------------+---------------------------+----------------+----------+------------+----------------+-------------+--------------+---------------+--------+-------------+----------+-------------+-----------+-----------

In [4]:
report_product_sales = (
    sales
    .groupBy(
        "product_name",
        "product_category",
    )
    .agg(
        F.countDistinct("sale_id").alias("sales_count"),
        F.sum("quantity").cast("long").alias("total_quantity_sold"),
        F.sum("total_price").cast("double").alias("total_revenue"),
        F.avg("product_rating").cast("double").alias("avg_rating"),
        F.sum("product_reviews").cast("long").alias("reviews_count"),
    )
    .withColumn(
        "quantity_rank",
        F.dense_rank().over(Window.orderBy(F.col("total_quantity_sold").desc()))
    )
    .withColumn(
        "category_revenue",
        F.sum("total_revenue").over(Window.partitionBy("product_category"))
    )
    .orderBy(F.col("quantity_rank"))
)

report_product_sales.show(20, truncate=False)
print(report_product_sales.count())

+------------+----------------+-----------+-------------------+------------------+------------------+-------------+-------------+-----------------+
|product_name|product_category|sales_count|total_quantity_sold|total_revenue     |avg_rating        |reviews_count|quantity_rank|category_revenue |
+------------+----------------+-----------+-------------------+------------------+------------------+-------------+-------------+-----------------+
|Dog Food    |Toy             |1152       |6346               |293603.40999999986|3.008593750000002 |581510       |1            |868101.6300000001|
|Bird Cage   |Toy             |1146       |6261               |296563.9199999999 |2.9839441535776627|567629       |2            |868101.6300000001|
|Bird Cage   |Cage            |1140       |6234               |279530.3700000004 |2.996842105263157 |584084       |3            |831117.9400000009|
|Dog Food    |Food            |1107       |6196               |283588.8799999999 |3.000813008130078 |549672     

In [5]:
customer_base = (
    sales
    .groupBy(
        "customer_id",
        "customer_first_name",
        "customer_last_name",
        "customer_email",
        "customer_country",
    )
    .agg(
        F.sum("total_price").cast("double").alias("total_purchase_amount"),
        F.countDistinct("sale_id").cast("long").alias("orders_count"),
        F.avg("total_price").cast("double").alias("avg_check"),
    )
)

customer_country_distribution = (
    customer_base
    .groupBy("customer_country")
    .agg(
        F.countDistinct("customer_id").cast("long").alias("customers_in_country")
    )
)

report_customer_sales = (
    customer_base
    .join(customer_country_distribution, "customer_country", "left")
    .withColumn(
        "purchase_rank",
        F.dense_rank().over(Window.orderBy(F.col("total_purchase_amount").desc()))
    )
    .select(
        "customer_id",
        "customer_first_name",
        "customer_last_name",
        "customer_email",
        "customer_country",
        "customers_in_country",
        "total_purchase_amount",
        "orders_count",
        "avg_check",
        "purchase_rank",
    )
    .orderBy(F.col("purchase_rank"))
)

report_customer_sales.show(20, truncate=False)
print(report_customer_sales.count())

+-----------+-------------------+------------------+----------------------------+----------------+--------------------+---------------------+------------+---------+-------------+
|customer_id|customer_first_name|customer_last_name|customer_email              |customer_country|customers_in_country|total_purchase_amount|orders_count|avg_check|purchase_rank|
+-----------+-------------------+------------------+----------------------------+----------------+--------------------+---------------------+------------+---------+-------------+
|4208       |Gus                |Hartshorn         |bfeasby57@youku.com         |Albania         |46                  |499.85               |1           |499.85   |1            |
|4364       |Hayes              |McKain            |sstappardbp@businessweek.com|Portugal        |336                 |499.8                |1           |499.8    |2            |
|2480       |Dawna              |Impey             |rivattspm@un.org            |Indonesia       |1174   

In [6]:
monthly_sales = (
    sales
    .groupBy("sale_year", "sale_month")
    .agg(
        F.countDistinct("sale_id").cast("long").alias("orders_count"),
        F.sum("quantity").cast("long").alias("total_quantity"),
        F.sum("total_price").cast("double").alias("monthly_revenue"),
        F.avg("total_price").cast("double").alias("avg_order_size"),
    )
)

yearly_sales = (
    sales
    .groupBy("sale_year")
    .agg(
        F.sum("total_price").cast("double").alias("yearly_revenue"),
        F.countDistinct("sale_id").cast("long").alias("yearly_orders"),
    )
)

time_window = Window.orderBy("sale_year", "sale_month")

report_time_sales = (
    monthly_sales
    .join(yearly_sales, "sale_year", "left")
    .withColumn(
        "previous_month_revenue",
        F.coalesce(F.lag("monthly_revenue").over(time_window), F.lit(0.0))
    )
    .withColumn(
        "revenue_diff_from_previous_month",
        F.col("monthly_revenue") - F.col("previous_month_revenue")
    )
    .select(
        "sale_year",
        "sale_month",
        "orders_count",
        "total_quantity",
        "monthly_revenue",
        "avg_order_size",
        "previous_month_revenue",
        "revenue_diff_from_previous_month",
        "yearly_revenue",
        "yearly_orders",
    )
    .orderBy("sale_year", "sale_month")
)

report_time_sales.show(20, truncate=False)
print(report_time_sales.count())

+---------+----------+------------+--------------+------------------+------------------+----------------------+--------------------------------+-----------------+-------------+
|sale_year|sale_month|orders_count|total_quantity|monthly_revenue   |avg_order_size    |previous_month_revenue|revenue_diff_from_previous_month|yearly_revenue   |yearly_orders|
+---------+----------+------------+--------------+------------------+------------------+----------------------+--------------------------------+-----------------+-------------+
|2021     |1         |874         |4856          |224158.54000000012|256.4743020594967 |0.0                   |224158.54000000012              |2529852.120000008|10000        |
|2021     |2         |739         |4070          |192348.31000000008|260.2818809201625 |224158.54000000012    |-31810.23000000004              |2529852.120000008|10000        |
|2021     |3         |843         |4561          |207282.20000000027|245.88635824436568|192348.31000000008    |1493

In [7]:
store_base = (
    sales
    .groupBy("store_name", "store_city", "store_country")
    .agg(
        F.sum("total_price").cast("double").alias("total_revenue"),
        F.sum("quantity").cast("long").alias("total_quantity"),
        F.countDistinct("sale_id").cast("long").alias("orders_count"),
        F.avg("total_price").cast("double").alias("avg_check"),
    )
)

store_city_distribution = (
    sales
    .groupBy("store_city")
    .agg(
        F.sum("total_price").cast("double").alias("city_revenue"),
        F.countDistinct("sale_id").cast("long").alias("city_orders_count"),
    )
)

store_country_distribution = (
    sales
    .groupBy("store_country")
    .agg(
        F.sum("total_price").cast("double").alias("country_revenue"),
        F.countDistinct("sale_id").cast("long").alias("country_orders_count"),
    )
)

report_store_sales = (
    store_base
    .join(store_city_distribution, "store_city", "left")
    .join(store_country_distribution, "store_country", "left")
    .withColumn(
        "store_revenue_rank",
        F.dense_rank().over(Window.orderBy(F.col("total_revenue").desc()))
    )
    .select(
        "store_name",
        "store_city",
        "store_country",
        "total_revenue",
        "total_quantity",
        "orders_count",
        "avg_check",
        "city_revenue",
        "city_orders_count",
        "country_revenue",
        "country_orders_count",
        "store_revenue_rank",
    )
    .orderBy("store_revenue_rank")
)

report_store_sales.show(20, truncate=False)
print(report_store_sales.count())


+-------------+---------------+-------------+-------------+--------------+------------+---------+-----------------+-----------------+------------------+--------------------+------------------+
|store_name   |store_city     |store_country|total_revenue|total_quantity|orders_count|avg_check|city_revenue     |city_orders_count|country_revenue   |country_orders_count|store_revenue_rank|
+-------------+---------------+-------------+-------------+--------------+------------+---------+-----------------+-----------------+------------------+--------------------+------------------+
|DabZ         |Grekan         |South Africa |499.85       |7             |1           |499.85   |739.23           |2                |20072.29          |75                  |1                 |
|Thoughtblab  |Fonte          |Poland       |499.8        |9             |1           |499.8    |499.8            |1                |81313.13000000005 |332                 |2                 |
|Camido       |Longzhong      |Swed

In [8]:
supplier_country_distribution = (
    sales
    .groupBy("supplier_country")
    .agg(
        F.sum("total_price").cast("double").alias("supplier_country_revenue"),
        F.countDistinct("sale_id").cast("long").alias("supplier_country_orders"),
    )
)

report_supplier_sales = (
    sales
    .groupBy("supplier_name", "supplier_country")
    .agg(
        F.sum("total_price").cast("double").alias("total_revenue"),
        F.avg("product_price").cast("double").alias("avg_product_price"),
        F.countDistinct("sale_id").cast("long").alias("orders_count"),
        F.sum("quantity").cast("long").alias("total_quantity"),
    )
    .join(supplier_country_distribution, "supplier_country", "left")
    .withColumn(
        "supplier_revenue_rank",
        F.dense_rank().over(Window.orderBy(F.col("total_revenue").desc()))
    )
    .select(
        "supplier_name",
        "supplier_country",
        "total_revenue",
        "avg_product_price",
        "orders_count",
        "total_quantity",
        "supplier_country_revenue",
        "supplier_country_orders",
        "supplier_revenue_rank",
    )
    .orderBy("supplier_revenue_rank")
)

report_supplier_sales.show(20, truncate=False)
print(report_supplier_sales.count())


+-------------+----------------+------------------+------------------+------------+--------------+------------------------+-----------------------+---------------------+
|supplier_name|supplier_country|total_revenue     |avg_product_price |orders_count|total_quantity|supplier_country_revenue|supplier_country_orders|supplier_revenue_rank|
+-------------+----------------+------------------+------------------+------------+--------------+------------------------+-----------------------+---------------------+
|Wikizz       |Indonesia       |4955.9400000000005|45.99857142857143 |14          |60            |265717.99000000005      |1079                   |1                    |
|Katz         |China           |3944.0499999999997|51.499333333333325|15          |80            |492823.3100000006       |1921                   |2                    |
|Wikizz       |China           |3684.54           |56.72571428571428 |14          |71            |492823.3100000006       |1921                   |3  

In [9]:
quality_base = (
    sales
    .groupBy(
        "product_name",
        "product_category",
    )
    .agg(
        F.avg("product_rating").cast("double").alias("avg_rating"),
        F.sum("product_reviews").cast("long").alias("reviews_count"),
        F.sum("quantity").cast("long").alias("total_quantity_sold"),
        F.countDistinct("sale_id").cast("long").alias("sales_count"),
    )
)

rating_sales_corr = (
    quality_base
    .select(
        F.corr("avg_rating", "total_quantity_sold").alias("rating_sales_correlation")
    )
)

report_product_quality = (
    quality_base
    .crossJoin(rating_sales_corr)
    .withColumn(
        "highest_rating_rank",
        F.dense_rank().over(Window.orderBy(F.col("avg_rating").desc()))
    )
    .withColumn(
        "lowest_rating_rank",
        F.dense_rank().over(Window.orderBy(F.col("avg_rating").asc()))
    )
    .withColumn(
        "reviews_rank",
        F.dense_rank().over(Window.orderBy(F.col("reviews_count").desc()))
    )
    .orderBy(F.col("highest_rating_rank"))
)

report_product_quality.show(20, truncate=False)
print(report_product_quality.count())

+------------+----------------+------------------+-------------+-------------------+-----------+------------------------+-------------------+------------------+------------+
|product_name|product_category|avg_rating        |reviews_count|total_quantity_sold|sales_count|rating_sales_correlation|highest_rating_rank|lowest_rating_rank|reviews_rank|
+------------+----------------+------------------+-------------+-------------------+-----------+------------------------+-------------------+------------------+------------+
|Cat Toy     |Toy             |3.0830324909747295|548517       |6028               |1108       |-0.34663522512968803    |1                  |9                 |7           |
|Dog Food    |Cage            |3.046574074074073 |522231       |5756               |1080       |-0.34663522512968803    |2                  |8                 |9           |
|Bird Cage   |Food            |3.0211069418386463|530547       |5710               |1066       |-0.34663522512968803    |3        

In [10]:
reports = {
    "report_product_sales": report_product_sales,
    "report_customer_sales": report_customer_sales,
    "report_time_sales": report_time_sales,
    "report_store_sales": report_store_sales,
    "report_supplier_sales": report_supplier_sales,
    "report_product_quality": report_product_quality,
}

for table_name, df in reports.items():
    print(f"Writing {table_name}: {df.count()} rows")

    (
        df.write
        .mode("overwrite")
        .format("jdbc")
        .option("url", CLICKHOUSE_URL)
        .option("dbtable", table_name)
        .option("user", CLICKHOUSE_PROPS["user"])
        .option("password", CLICKHOUSE_PROPS["password"])
        .option("driver", CLICKHOUSE_PROPS["driver"])
        .option("createTableOptions", "ENGINE = MergeTree() ORDER BY tuple()")
        .save()
    )

print("ClickHouse reports written.")

Writing report_product_sales: 9 rows
Writing report_customer_sales: 10000 rows
Writing report_time_sales: 12 rows
Writing report_store_sales: 10000 rows
Writing report_supplier_sales: 6295 rows
Writing report_product_quality: 9 rows
ClickHouse reports written.


In [11]:
spark.stop()